# RAG Retrieval Test Notebook

This notebook tests the document retrieval functionality of the RAG system.

## RAG Flow Overview

```
Query → Embedding Service → PgVector Similarity Search → Tenant Validation → Results
```

**Key Components:**
- `RAGService.query_knowledge_base()`: Main retrieval method
- `EmbeddingService`: Converts text to embeddings (all-MiniLM-L6-v2, 384 dimensions)
- `PGVector`: PostgreSQL + pgvector for similarity search
- **Tenant Isolation**: All queries filtered by `tenant_id` in metadata

## Setup

In [1]:
import sys
sys.path.insert(0, '.')

from src.services.rag_service import get_rag_service
from pprint import pprint
import json

c:\Users\gensh\Downloads\ITL_chatbot\itl_chatbot_works\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [ ]:
# Test Configuration
TENANT_ID = "3105b788-b5ff-4d56-88a9-532af4ab4ded"  # Your tenant ID
QUERY = "hướng dẫn quản lý danh sách đối tác"  # Your test query
TOP_K = 5  # Number of results to retrieve

## Initialize RAG Service

In [4]:
# Get RAG service instance
rag_service = get_rag_service()

print(f"RAG Service initialized")
print(f"Embedding Model: {rag_service.embedding_service.model_name}")
print(f"Embedding Dimension: {rag_service.embedding_service.dimension}")
print(f"Collection Name: {rag_service.collection_name}")

{"model_name": "all-MiniLM-L6-v2", "event": "embedding_service_initializing", "level": "info", "timestamp": "2025-11-27T07:41:26.242877Z"}


Use pytorch device_name: cpu
Load pretrained SentenceTransformer: all-MiniLM-L6-v2


{"model_name": "all-MiniLM-L6-v2", "dimension": 384, "event": "embedding_service_initialized", "level": "info", "timestamp": "2025-11-27T07:41:32.618476Z"}
{"model_name": "all-MiniLM-L6-v2", "dimension": 384, "event": "embedding_service_singleton_created", "level": "info", "timestamp": "2025-11-27T07:41:32.620493Z"}
{"chunk_size": 800, "chunk_overlap": 200, "overlap_percentage": "25.0%", "event": "document_processor_initialized", "level": "info", "timestamp": "2025-11-27T07:41:32.622513Z"}
{"event": "document_processor_singleton_created", "level": "info", "timestamp": "2025-11-27T07:41:32.624527Z"}
{"backend": "pgvector", "embedding_model": "all-MiniLM-L6-v2", "embedding_dimension": 384, "collection_name": "knowledge_documents", "event": "rag_service_initialized", "level": "info", "timestamp": "2025-11-27T07:41:32.629002Z"}
{"event": "rag_service_singleton_created", "level": "info", "timestamp": "2025-11-27T07:41:32.631013Z"}
RAG Service initialized
Embedding Model: all-MiniLM-L6-v2
Em

## Test 1: Basic Retrieval Query

Test the basic document retrieval functionality.

In [5]:
print("=" * 60)
print("TEST 1: BASIC RETRIEVAL")
print("=" * 60)
print(f"Query: {QUERY}")
print(f"Tenant ID: {TENANT_ID}")
print(f"Top K: {TOP_K}")
print()

# Execute query
result = rag_service.query_knowledge_base(
    tenant_id=TENANT_ID,
    query=QUERY,
    top_k=TOP_K
)

# Display results
print(f"Success: {result.get('success')}")
print(f"Total Results: {result.get('total_results')}")
print()

TEST 1: BASIC RETRIEVAL
Query: tạo báo giá thuê ngoài FCL
Tenant ID: 3105b788-b5ff-4d56-88a9-532af4ab4ded
Top K: 5



Batches: 100%|██████████| 1/1 [00:00<00:00,  6.61it/s]


{"tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded", "collection_name": "knowledge_documents", "query_length": 26, "results_count": 5, "section_filter": null, "include_section_context": true, "event": "knowledge_base_queried", "level": "info", "timestamp": "2025-11-27T07:41:37.500497Z"}
Success: True
Total Results: 5



## Test 2: Examine Retrieved Documents

Inspect the structure and content of retrieved documents.

In [6]:
print("=" * 60)
print("TEST 2: DOCUMENT DETAILS")
print("=" * 60)

documents = result.get('documents', [])

for i, doc in enumerate(documents):
    print(f"\n--- Document {i+1} ---")
    print(f"Rank: {doc.get('rank')}")
    print(f"Distance (Cosine): {doc.get('distance'):.4f}")
    print(f"\nMetadata:")
    metadata = doc.get('metadata', {})
    print(f"  - Tenant ID: {metadata.get('tenant_id')}")
    print(f"  - Document Name: {metadata.get('document_name')}")
    print(f"  - Section: {metadata.get('section_title', 'N/A')}")
    print(f"  - Section Number: {metadata.get('section_number', 'N/A')}")
    print(f"  - File Type: {metadata.get('file_type')}")
    print(f"  - Page: {metadata.get('page', 'N/A')}")
    print(f"  - Chunk Index: {metadata.get('chunk_index')} / {metadata.get('chunk_total')}")
    print(f"\nContent Preview:")
    content = doc.get('content', '')
    print(f"  {content[:200]}..." if len(content) > 200 else f"  {content}")
    print("-" * 60)

TEST 2: DOCUMENT DETAILS

--- Document 1 ---
Rank: 1
Distance (Cosine): 0.2364

Metadata:
  - Tenant ID: 3105b788-b5ff-4d56-88a9-532af4ab4ded
  - Document Name: string
  - Section: N/A
  - Section Number: N/A
  - File Type: .txt
  - Page: N/A
  - Chunk Index: 1025 / 1431

Content Preview:
  5.1. Tạo bảng giá thuê cont/ remooc từ Nhà thầu phụ (Chỉ áp dụng cho BU là FTL)
5.1.1. Bảng giá thuê Container
Mục đích: Cho phép người dùng tạo và quản lý bảng giá thuê cont từ nhà thầu phụ
Đường dẫn...
------------------------------------------------------------

--- Document 2 ---
Rank: 2
Distance (Cosine): 0.2657

Metadata:
  - Tenant ID: 3105b788-b5ff-4d56-88a9-532af4ab4ded
  - Document Name: string
  - Section: N/A
  - Section Number: N/A
  - File Type: .txt
  - Page: N/A
  - Chunk Index: 280 / 1431

Content Preview:
  4.2. Tạo giá mua
Giá mua (Thuê ngoài/ Hire) là giá khi BU (Business Unit) đi thuê đối tác bên ngoài vận chuyển đơn hàng cho khách hàng
4.2.1. FCL (FCL Buying Price)
Mục đích: Ch

## Test 3: Formatted Content for LLM

Show how documents are formatted for LLM consumption with section context.

In [12]:
print("=" * 60)
print("TEST 3: FORMATTED CONTENT (LLM-Ready)")
print("=" * 60)

for i, doc in enumerate(documents[:3]):  # Show first 3
    print(f"\n--- Document {i+1} (Formatted) ---")
    formatted = doc.get('formatted_content', doc.get('content'))
    print(formatted)
    print("-" * 60)

TEST 3: FORMATTED CONTENT (LLM-Ready)

--- Document 1 (Formatted) ---
[Section: Not specified]

Hình ảnh : Tạo Yêu cầu sửa chữa khi đã có danh sách xe cần sửa chữa (1)
• Bước 2: Chọn vào chức năng Tạo mới Yêu cầu bảo dưỡng sửa chữa
• Bước 3: Người dùng nhập các thông tin bắt buộc về M/R Vehicle Type/ Loại phương tiện bảo dưỡng sửa chữa, Maintenance Type/ Loại bảo dưỡng và M/R Place/ Nơi bảo dưỡng sửa chữa tại Step 1 này
------------------------------------------------------------

--- Document 2 (Formatted) ---
[Section: Not specified]

7.2. Yêu Cầu Bảo Dưỡng Sửa chữa
Mục đích: Quản lý các Yêu cầu bảo dưỡng sửa chữa (YCBDSC). Mỗi Yêu cầu bảo dưỡng sửa chữa sẽ ứng với một lệnh điều động cho Tài xế thực hiện mang xe đi bảo dưỡng sửa chữa (BDSC)
Đường dẫn: eTMS : Bảo dưỡng và Sửa chữa a Yêu cầu sửa chữa
------------------------------------------------------------

--- Document 3 (Formatted) ---
[Section: Not specified]

7.4. Yêu cầu thanh toán Bảo Dưỡng Sửa chữa
Mục đích: Quản lý tạo và g

## Test 4: Different Query

Test with a different query to see how results change.

In [7]:
# Try a different query
QUERY_2 = "Cập nhật giá dầu"  # Change this to your query

print("=" * 60)
print("TEST 4: DIFFERENT QUERY")
print("=" * 60)
print(f"Query: {QUERY_2}")
print()

result_2 = rag_service.query_knowledge_base(
    tenant_id=TENANT_ID,
    query=QUERY_2,
    top_k=3
)

print(f"Total Results: {result_2.get('total_results')}")
print()

for i, doc in enumerate(result_2.get('documents', [])):
    print(f"\nDocument {i+1}:")
    print(f"  Distance: {doc.get('distance'):.4f}")
    print(f"  Section: {doc.get('metadata', {}).get('section_title', 'N/A')}")
    print(f"  Content: {doc.get('content')[:150]}...")
    print("-" * 40)

TEST 4: DIFFERENT QUERY
Query: Cập nhật giá dầu



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{"tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded", "collection_name": "knowledge_documents", "query_length": 16, "results_count": 3, "section_filter": null, "include_section_context": true, "event": "knowledge_base_queried", "level": "info", "timestamp": "2025-11-27T02:21:17.110524Z"}
Total Results: 3


Document 1:
  Distance: 0.1507
  Section: 4.12.1. Cập nhật giá dầu
  Content: Hình ảnh : Cập nhật giá dầu (2)...
----------------------------------------

Document 2:
  Distance: 0.1544
  Section: 4.12.1. Cập nhật giá dầu
  Content: Hình ảnh : Cập nhật giá dầu (1)...
----------------------------------------

Document 3:
  Distance: 0.1748
  Section: 4.12.1. Cập nhật giá dầu
  Content: 4.12.1. Cập nhật giá dầu...
----------------------------------------


## Test 5: Collection Statistics

Check how many documents are in the knowledge base for this tenant.

In [8]:
print("=" * 60)
print("TEST 5: COLLECTION STATISTICS")
print("=" * 60)

stats = rag_service.get_collection_stats(tenant_id=TENANT_ID)

print(f"Success: {stats.get('success')}")
print(f"Tenant ID: {stats.get('tenant_id')}")
print(f"Collection Name: {stats.get('collection_name')}")
print(f"Total Documents (Chunks): {stats.get('document_count')}")
print()

if stats.get('success'):
    print(f"✅ Knowledge base is ready with {stats.get('document_count')} chunks")
else:
    print(f"❌ Error: {stats.get('error')}")

TEST 5: COLLECTION STATISTICS
{"tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded", "collection_name": "knowledge_documents", "document_count": 4706, "event": "collection_stats_retrieved", "level": "info", "timestamp": "2025-11-27T02:21:30.778450Z"}
Success: True
Tenant ID: 3105b788-b5ff-4d56-88a9-532af4ab4ded
Collection Name: knowledge_documents
Total Documents (Chunks): 4706

✅ Knowledge base is ready with 4706 chunks


## Test 6: Section Filtering (Optional)

Test retrieval with section filtering if you know a specific section.

In [9]:
# Optional: Filter by section
SECTION_FILTER = "4.5.1"  # Change to your section number or title

print("=" * 60)
print("TEST 6: SECTION FILTERING")
print("=" * 60)
print(f"Query: {QUERY}")
print(f"Section Filter: {SECTION_FILTER}")
print()

result_filtered = rag_service.query_knowledge_base(
    tenant_id=TENANT_ID,
    query=QUERY,
    top_k=5,
    section_filter=SECTION_FILTER
)

print(f"Total Results: {result_filtered.get('total_results')}")
print()

for i, doc in enumerate(result_filtered.get('documents', [])):
    metadata = doc.get('metadata', {})
    print(f"\nDocument {i+1}:")
    print(f"  Section: {metadata.get('section_number')} - {metadata.get('section_title')}")
    print(f"  Distance: {doc.get('distance'):.4f}")
    print(f"  Content: {doc.get('content')[:100]}...")
    print("-" * 40)

TEST 6: SECTION FILTERING
Query: Hướng dẫn tạo bảng giá fcl?
Section Filter: 4.5.1



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{"tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded", "collection_name": "knowledge_documents", "query_length": 27, "results_count": 5, "section_filter": "4.5.1", "include_section_context": true, "event": "knowledge_base_queried", "level": "info", "timestamp": "2025-11-27T02:21:44.356970Z"}
Total Results: 5


Document 1:
  Section: 4.5.1 - 4.5.1. Bảng giá FCL (FCL Rate Card List)
  Distance: 0.1476
  Content: Tạo mới bảng giá FCL:...
----------------------------------------

Document 2:
  Section: 4.5.1 - 4.5.1. Bảng giá FCL (FCL Rate Card List)
  Distance: 0.1921
  Content: Tạo bảng giá bán FCL theo ngày trong tuần:...
----------------------------------------

Document 3:
  Section: 4.5.1 - 4.5.1. Bảng giá FCL (FCL Rate Card List)
  Distance: 0.2041
  Content: Hình ảnh : Tạo mới bảng giả FCL theo ngày trong tuần (1)...
----------------------------------------

Document 4:
  Section: 4.5.1 - 4.5.1. Bảng giá FCL (FCL Rate Card List)
  Distance: 0.2046
  Content: Hình ảnh : Tạo mới bảng 

## Test 7: Full JSON Output

View the complete JSON structure of the retrieval result.

In [10]:
print("=" * 60)
print("TEST 7: FULL JSON OUTPUT")
print("=" * 60)

# Pretty print the full result
print(json.dumps(result, indent=2, ensure_ascii=False, default=str))

TEST 7: FULL JSON OUTPUT
{
  "success": true,
  "tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded",
  "query": "Hướng dẫn tạo bảng giá fcl?",
  "documents": [
    {
      "content": "Tạo bảng giá FCL",
      "metadata": {
        "page": 235,
        "style": "List Paragraph",
        "doc_id": "e78f2db3-266b-4682-abeb-63e6e184bba8",
        "source": "document",
        "file_type": ".docx",
        "tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded",
        "is_heading": false,
        "chunk_index": 1792,
        "chunk_total": 4706,
        "ingested_at": "2025-11-27T01:53:38.308117",
        "document_name": "string",
        "section_title": "* Tên dự án",
        "source_detail": "upload_document",
        "section_number": null,
        "paragraph_index": 2347,
        "original_filename": "eTMS.docx",
        "uploaded_by_admin": null
      },
      "distance": 0.10174810886383057,
      "rank": 1,
      "formatted_content": "[Section: * Tên dự án]\n\nTạo bảng giá FCL",
    

## Summary

This notebook demonstrates:

1. ✅ **Basic Retrieval**: Query → Documents with similarity scores
2. ✅ **Document Structure**: Metadata (tenant_id, section, file info, chunk info)
3. ✅ **LLM-Ready Format**: Formatted content with section context
4. ✅ **Multiple Queries**: Testing different search terms
5. ✅ **Collection Stats**: Total document count
6. ✅ **Section Filtering**: Narrow results to specific sections
7. ✅ **Full JSON**: Complete API response structure

### Key Findings:

- **Tenant Isolation**: All results are filtered by `tenant_id`
- **Cosine Distance**: Lower distance = more similar (0 = identical)
- **Metadata Rich**: Each chunk has section info, page numbers, file details
- **Section Context**: Results include formatted content for LLM consumption